In [1]:
# infer_parameters.py
from __future__ import annotations
from copy import deepcopy
from pathlib import Path
from typing import Sequence, Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt 

# utilities
from bioprocess_utils import *   

In [2]:
inputs = load_model_inputs("./inputs/inputs.json")
prms_to_infer = ("mu_max","K_subs","K_L_a")
csv_path="./simulated_dataset/pputida_fedbatch_v1.wide.csv"
observables = ("Biomass","Sub")
# making noise in the base inputs
for prm_i in prms_to_infer:
    inputs["params"][prm_i]["values"][0] = inputs["params"][prm_i]["values"][0]*5.0


In [ ]:

# Fit μmax, Ks, KLa from BIOMASS across all experiments in the CSV
report = fit_parameters(
    csv_path=csv_path,
    inputs = inputs,
    observables=observables,
    param_names=prms_to_infer,
    initial_names=(),           # to also infer Biomass0: initial_names=("Biomass",)
    experiment_ids=None,        # None -> use all experiments; or pass e.g. [0]
    bounds=None,                # None -> auto ±10× around seeds
    x0=None,                    # None -> seeds from inputs.json
    verbose=2,
)
print("\n=== Fit summary ===")
for name, val in zip(report["theta_names"], report["theta_opt"]):
    print(f"{name:10s} = {val:.6g}")
print("success:", report["success"], "| cost:", report["cost"])

In [ ]:
fitted_inputs = report["fitted_inputs"]

fig, axes = plot_dataset_with_traces(
    csv_path=csv_path,
    inputs_init=inputs,
    inputs_fit=fitted_inputs,
    observables=observables,      
    envelope="std",)